In [ ]:
# Установка/импорт библиотек и подключение к MinIO
!pip -q install minio pandas pyarrow

from minio import Minio
from minio.error import S3Error
import pandas as pd

In [ ]:
# Параметры подключения
MINIO_ENDPOINT = "localhost:9000"
MINIO_ACCESS_KEY = "admin" # из переменных окружения
MINIO_SECRET_KEY = "admin123" # из переменных окружения
MINIO_SECURE = False

In [ ]:
# Создание клиента
client = Minio(
    MINIO_ENDPOINT,
    access_key=MINIO_ACCESS_KEY,
    secret_key=MINIO_SECRET_KEY,
    secure=MINIO_SECURE,
)

# Проверка подключения
try:
    buckets = client.list_buckets()
    print("Подключение успешно")
    print("Доступные бакеты:", [b.name for b in buckets])
except S3Error as e:
    print("Ошибка подключения:", e)

In [ ]:
# Создание бакета
bucket_name = "hw-minIo"

if not client.bucket_exists(bucket_name):
    client.make_bucket(bucket_name)
    print(f"Бакет '{bucket_name}' создан")
else:
    print(f"Бакет '{bucket_name}' уже существует")

In [ ]:
# Создание тестовых данных
df = pd.DataFrame(
    {
        "student_id": [101, 102, 103, 104, 105],
        "first_name": ["Иван", "Мария", "Алексей", "Елена", "Дмитрий"],
        "last_name": ["Петров", "Сидорова", "Иванов", "Кузнецова", "Смирнов"],
        "course": ["Математика", "Физика", "Информатика", "Химия", "Биология"],
        "grade": [4.8, 4.5, 5.0, 4.3, 4.7],
        "enrollment_year": [2022, 2021, 2023, 2022, 2021]
    }
)

print("Созданный DataFrame:")
print(df)
print(f"\nРазмер данных: {df.shape[0]} строк, {df.shape[1]} столбцов")

In [ ]:
# Сохранение в формате CSV
from io import BytesIO

object_name_csv = "students/students_data.csv"

# Конвертация DataFrame в CSV
csv_bytes = df.to_csv(index=False, encoding='utf-8-sig').encode("utf-8-sig")
csv_buffer = BytesIO(csv_bytes)

# Загрузка в MinIO
client.put_object(
    bucket_name=bucket_name,
    object_name=object_name_csv,
    data=csv_buffer,
    length=len(csv_bytes),
    content_type="text/csv; charset=utf-8-sig",
)

print(f"CSV файл загружен: s3://{bucket_name}/{object_name_csv}")
print(f"Размер файла: {len(csv_bytes)} байт")

In [ ]:
# Чтение CSV файла из MinIO
resp = client.get_object(bucket_name, object_name_csv)
try:
    downloaded = resp.read()
finally:
    resp.close()
    resp.release_conn()

# Преобразование обратно в DataFrame
df_csv_back = pd.read_csv(BytesIO(downloaded), encoding='utf-8-sig')

print("Загруженные данные из CSV:")
print(df_csv_back)
print(f"\nПроверка: данные идентичны? {df.equals(df_csv_back)}")

In [ ]:
# Сохранение в формате Parquet
object_name_parquet = "students/students_data.parquet"

# Конвертация DataFrame в Parquet
parquet_buffer = BytesIO()
df.to_parquet(parquet_buffer, index=False, compression='snappy')
parquet_bytes = parquet_buffer.getvalue()

# Загрузка в MinIO
client.put_object(
    bucket_name=bucket_name,
    object_name=object_name_parquet,
    data=BytesIO(parquet_bytes),
    length=len(parquet_bytes),
    content_type="application/octet-stream",
)

print(f"\nParquet файл загружен: s3://{bucket_name}/{object_name_parquet}")
print(f"Размер файла: {len(parquet_bytes)} байт")
print(f"Сжатие: Parquet ({len(parquet_bytes)} байт) vs CSV ({len(csv_bytes)} байт)")

In [ ]:
# Чтение Parquet файла из MinIO
resp = client.get_object(bucket_name, object_name_parquet)
try:
    downloaded = resp.read()
finally:
    resp.close()
    resp.release_conn()

# Преобразование обратно в DataFrame
df_parquet_back = pd.read_parquet(BytesIO(downloaded))

print("\nЗагруженные данные из Parquet:")
print(df_parquet_back)
print(f"\nПроверка на идентичность данных {df.equals(df_parquet_back)}")

In [ ]:
# Проверка содержимого бакета
print("\nСодержимое бакета:")
objects = client.list_objects(bucket_name, recursive=True)
for obj in objects:
    print(f" {obj.object_name} ({obj.size} байт, {obj.last_modified})")

In [ ]:
# Информация о подключении (для проверки)
print("\n" + "="*50)
print("ИНФОРМАЦИЯ О ПОДКЛЮЧЕНИИ:")
print(f"MinIO Endpoint: {MINIO_ENDPOINT}")
print(f"Bucket: {bucket_name}")
print(f"Доступные файлы:")
print(f"  1. {object_name_csv}")
print(f"  2. {object_name_parquet}")
print(f"Web Console: http://localhost:9001")
print("="*50)